# Логистическая регрессия с нуля: градиентный спуск, SGD и elastic net

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ignat2008/hw_2_linear_models/blob/main/logistic_regression_sgd.ipynb)

В этом проекте я разбираю линейные алгоритмы на уровне численных методов:

1. градиентный спуск на простой функции двух переменных;
2. генератор батчей для стохастического градиентного спуска;
3. собственный класс логистической регрессии для бинарной классификации;
4. логистическая регрессия с $l_1$- и $l_2$-регуляризацией (elastic net);
5. проверка на синтетических данных и на цифрах MNIST (классы 0 и 1).

> **Статус.** В ячейках с пометками `YOUR CODE` / `<YOUR CODE>` код пока не дописан, поэтому ноутбук целиком не запускается. Ячейки с `assert` показывают ожидаемые значения для проверки.

In [ ]:
import numpy as np
import pandas as pd
import scipy.linalg as sla
import matplotlib.pyplot as plt

from sklearn import datasets
from sklearn.linear_model import LinearRegression, Lasso, Ridge

In [ ]:
np.random.seed(42)

## 1. Градиентный спуск на простой функции

Работаю с функцией двух переменных

$$f(\vec{r}) = f(x, y) = \sin^2 x + \sin^2 y.$$

In [ ]:
def f(r):
    """
    :param r: np.array(np.float) вектор длины 2
    :return: np.float

    На вход функции мы подаем двумерный массив: r = [x,y]
    Используем функцию np.sin() (https://numpy.org/doc/stable/reference/generated/numpy.sin.html) и оператор возведения в степень ** (https://numpy.org/doc/stable/reference/generated/numpy.power.html), которые применяются к массивам поэлементно
    """
    return np.sum(np.sin(r)**2)

Цель оптимизации — найти минимум функции. В машинном обучении минимизируют **функцию потерь**, например [MSE](https://scikit-learn.org/stable/modules/model_evaluation.html#mean-squared-error), то есть ищут точку $(x_0, y_0)$, в которой значение минимально. Один из методов минимизации — **градиентный спуск**. Здесь $r$ — numpy-вектор длины 2.

### Градиент

**Градиент** — вектор, который указывает направление наибольшего возрастания функции. Для функции двух переменных:

$$\nabla f(x, y) = \left( \dfrac{\partial f}{\partial x}, \dfrac{\partial f}{\partial y} \right),$$

где $\dfrac{\partial f}{\partial x}$ и $\dfrac{\partial f}{\partial y}$ — частные производные по $x$ и $y$.

*Пример.* Для $f(x, y) = x^2 + 3xy + y^2$:

$$\nabla f(x, y) = \left(2x + 3y,\; 2y + 3x\right).$$

Производные считаю **аналитически** и переношу в код, без численного дифференцирования: для нашей функции $\dfrac{\partial f}{\partial x} = \sin 2x$, $\dfrac{\partial f}{\partial y} = \sin 2y$.

In [ ]:
# Сначала реализуем функцию, вычисляющую градиент

def grad_f(r):
    """
    Градиент функциии f, определенной выше.
    :param r: np.array[2]: float вектор длины 2
    :return: np.array[2]: float вектор длины 2
    """
    return np.sin(2*r)

In [ ]:
# Проверим, что градиент принимает вектор из двух чисел и выдает на этой точке верное значение
assert np.allclose(grad_f(np.array([1, 2])),
                   np.array([0.90929743, -0.7568025])), "Что-то не так!"

In [ ]:
def grad_descent_2d(f, grad_f, lr, num_iter=100, r0=None):
    """
    функция, которая реализует градиентный спуск для функции f от двух переменных.
        :param f: скалярная функция двух переменных
        :param grad_f: функция, возвращающая градиент функции f (устроена так, как реализованная вами выше grad_f)
        :param lr: learning rate алгоритма
        :param num_iter: количество итераций градиентного спуска
        :param r0: начальное значение аргумента f(r), которое мы инициируем случайно или можем передать как аргумент в функцию grad_descent_2d()

        :return: np.array[num_iter, 2] пары вида (r, f(r))
    """
    if r0 is None:
        r0 = np.random.random(2)

    # в процессе градиентного спуска будем сохранять в переменную history значения
    history = []

    # итерация цикла -- шаг градиентного спуска
    curr_r = r0.copy()
    for iter_num in range(num_iter):
        entry = np.hstack((curr_r, f(curr_r)))
        history.append(entry)

        # curr_r - текущие значения аргумента в процессе градиентного спуска
        curr_r -=

    return np.vstack(history)

### Проверка работы спуска

In [ ]:
steps = grad_descent_2d(f, grad_f, lr=0.1, num_iter=20)

Траекторию спуска рисую на 3D-графике функции ([plotly](https://plotly.com/python/), график можно вращать и приближать). Звёздочки — точки $(x, y, f(x, y))$, по которым двигался алгоритм. Если спуск реализован верно, они сходятся к одному из минимумов функции. Менять можно начальное приближение, `lr` и `num_iter`.

In [ ]:
# Посмотрим, что из себя предствавляет вывод функции:
steps

# при желании траекторию можно вывести таблицей:
#pd.DataFrame(data = steps, columns=['x', 'y', 'f([x,y])'])

In [ ]:
import numpy as np
import plotly.graph_objects as go

# Создание сетки
border = 1.5 # Меняйте значения border, чтобы увеличивать или уменьшать область
X, Y = np.meshgrid(np.linspace(-border, border, 100), np.linspace(-border, border, 100))
Z = np.array([f(np.array([x, y])) for x, y in zip(np.ravel(X), np.ravel(Y))]).reshape(X.shape)

# Создание 3D-графика
fig = go.Figure(data=[
    go.Surface(z=Z, x=X, y=Y, colorscale='Viridis')
])

# Добавление линии и маркеров
fig.add_trace(go.Scatter3d(
    x=steps[:, 0],
    y=steps[:, 1],
    z=steps[:, 2],
    mode='lines+markers',
    marker=dict(size=8, color='red', symbol='cross'),
    line=dict(color='black', width=5)
))

# Настройка осей
fig.update_layout(
    scene=dict(
        xaxis_title='X',
        yaxis_title='Y',
        zaxis_title='Z',
        zaxis=dict(range=[-1, 5]),  # Установка пределов для Z
        aspectratio=dict(x=1, y=1, z=0.7)  # Соотношение сторон
    ),
    margin=dict(l=0, r=0, b=0, t=0)  # Удаление отступов
)

# Отображение графика
fig.show();

In [ ]:
# %matplotlib osx

from mpl_toolkits.mplot3d import Axes3D
import matplotlib.pyplot as plt
from matplotlib import cm
import numpy as np

path = []

X, Y = np.meshgrid(np.linspace(-3, 3, 100), np.linspace(-3, 3, 100))

fig = plt.figure(figsize=(16, 10))
ax = fig.gca(projection='3d')

zs = np.array([f(np.array([x,y]))
              for x, y in zip(np.ravel(X), np.ravel(Y))])
Z = zs.reshape(X.shape)


ax.plot_surface(X, Y, Z, cmap=cm.coolwarm, zorder=2)

ax.plot(xs=steps[:, 0], ys=steps[:, 1], zs=steps[:, 2],
        marker='*', markersize=20, zorder=3,
        markerfacecolor='y', lw=3, c='black')

ax.set_zlim(0, 5)
ax.view_init(elev=60)
plt.show()

График значения функции по шагам спуска:

In [ ]:
plt.figure(figsize=(14,7))
plt.xlabel('grad descent step number')
plt.ylabel('$f(r)$')
plt.title('Значение функции на каждом шаге градиентного спуска.')

f_values = list(map(lambda x: x[2], steps))
plt.plot(f_values, label='gradient descent result')
plt.legend()

Всегда нужно проверять, сошёлся ли градиентный спуск. Для большинства моделей 3D-график построить не получится, поэтому следим за тем, как менялся лосс на каждом шаге.

## 2. Генератор батчей

Вместо случайных `batch_size` строк на каждом шаге обучение делят на **эпохи**: некоторые строки при случайном выборе попадают в батчи заметно реже других, а в эпохе каждый пример должен встретиться ровно один раз. Для этого датасет перемешивают и каждый раз берут очередные `batch_size` строк.

Если размер датасета не делится на `batch_size`, последний неполный батч отбрасывается.

Генератор принимает признаки, ответы и размер батча и возвращает кортежи $(X_{batch}, y_{batch})$. Чтобы не перемешивать сам датасет, используется `np.random.permutation(len(X))`: индексы батчей берутся из перемешанной последовательности `perm` (см. [документацию numpy по индексации](https://numpy.org/doc/stable/user/basics.indexing.html#integer-array-indexing)). О генераторах Python: [документация](https://wiki.python.org/moin/Generators).

In [ ]:
def generate_batches(X, y, batch_size):
    """
    param X: np.array[n_objects, n_features] --- матрица объекты-признаки
    param y: np.array[n_objects] --- вектор целевых переменных
    """
    assert len(X) == len(y)
    np.random.seed(42)
    X = np.array(X)
    y = np.array(y)
    perm = np.random.permutation(len(X))

    for batch_start in <YOUR CODE>:
        # YOUR CODE
        yield # YOUR CODE

### Проверка

Генератор должен перемешивать датасет, и каждый элемент должен встречаться ровно один раз:

In [ ]:
X_fake = np.arange(100)[:, np.newaxis]
y_fake = np.arange(100) + 1000

X_reconstructed, y_reconstructed = [], []
for X_batch, y_batch in generate_batches(X_fake, y_fake, 10):
    X_reconstructed.append(X_batch)
    y_reconstructed.append(y_batch)

X_reconstructed = np.concatenate(X_reconstructed)
y_reconstructed = np.concatenate(y_reconstructed)

assert (X_fake != X_reconstructed).all(), "Что-то не так!"
assert (y_fake != y_reconstructed).all(), "Что-то не так!"

assert (np.sort(X_reconstructed, axis=0) == X_fake).all(), "Что-то не так!"

Теперь `batch_size`, не делящий размер датасета: неполный батч должен быть отброшен.

In [ ]:
X_fake = np.arange(100)[:, np.newaxis]
y_fake = np.arange(100) + 1000

num_batches = 0
for X_batch, y_batch in generate_batches(X_fake, y_fake, 7):
    num_batches += 1

assert num_batches == len(X_fake) // 7, "Что-то не так!"

## 3. Логистическая регрессия на SGD

Класс логистической регрессии без регуляризации, обучаемый стохастическим градиентным спуском на батчах из генератора выше. Нужно реализовать методы `fit` и `get_grad` класса `MyLogisticRegression`.

Функция потерь и её градиент:

$$Loss(y, p) = -\sum_{i=1}^{\ell} (y_i \log (p_i) + (1 - y_i) \log (1 - p_i))$$

$$\frac{\partial L}{\partial w} = X^T (p - y)$$

In [ ]:
def logit(x, w):
    return np.dot(x, w)

def sigmoid(h):
    return 1. / (1 + np.exp(-h))

In [ ]:
class MyLogisticRegression(object):
    def __init__(self):
        self.w = None

    def fit(self, X, y, epochs=10, lr=0.1, batch_size=100):
        l, n = X.shape
        if self.w is None:
            np.random.seed(42)
            # Вектор столбец в качестве весов
            self.w = np.random.randn(n + 1)

        X_train = np.concatenate((np.ones((l, 1)), X), axis=1)

        losses = []

        # Положите в лист losses лосс на каждом батче. Не нужно усреднять лосс по эпохе.

        for i in range(epochs):
            for X_batch, y_batch in generate_batches(X_train, y, batch_size):
                # В X_train уже добавлен вектор 1

                predictions = # YOUR CODE: предскажите вероятности на X_batch
                loss = # YOUR CODE: вычислите loss на текущем батче

                assert (np.array(loss).shape == tuple()), "Лосс должен быть скаляром!"

                losses.append(<YOUR CODE>)

                # YOUR CODE: обновите self.w по формуле градиентного спуска. Используйте функцию self.get_grad для вычисления градиента. Не забудьте про learning rate!

        return losses

    def get_grad(self, X_batch, y_batch, predictions):
        """
        param X_batch: np.array[batch_size, n_features + 1] --- матрица объекты-признаки
        param y_batch: np.array[batch_size] --- батч целевых переменных
        param predictions: np.array[batch_size] --- батч вероятностей классов

        Принимает на вход X_batch с уже добавленной колонкой единиц.
        Выдаёт градиент функции потерь логистической регрессии
        как сумму градиентов функции потерь на всех объектах батча
        ВНИМАНИЕ! Нулевая координата вектора весов -- это BIAS, а не вес признака.
        Также не нужно ДЕЛИТЬ ГРАДИЕНТ НА РАЗМЕР БАТЧА:
        нас интересует не среднее, а сумма.
        В качестве оператора умножения матриц можно использовать @

        Выход -- вектор-столбец градиентов для каждого веса (np.array[n_features + 1])
        """

        # компонент градиента из логрегрессии
        # следите за размерностями

        grad_basic = # YOUR CODE
        assert grad_basic.shape == (X_batch.shape[1],) , "Градиенты должны быть столбцом из k_features + 1 элементов"

        return grad_basic

    def predict_proba(self, X):
        l, n = X.shape
        X_ = np.concatenate((np.ones((l, 1)), X), axis=1)
        return sigmoid(logit(X_, self.w))

    def _predict_proba_internal(self, X):
        """
        Возможно, вы захотите использовать эту функцию вместо predict_proba, поскольку
        predict_proba конкатенирует вход с вектором из единиц, что не всегда удобно
        для внутренней логики вашей программы
        """
        return sigmoid(logit(X, self.w))

    def predict(self, X, threshold=0.5):
        return self.predict_proba(X) >= threshold

    def get_weights(self):
        return self.w.copy()
        # copy тут используется неспроста. Если copy не использовать, то get_weights()
        # выдаст ссылку на объект, а, значит, модифицируя результат применения функции
        # get_weights(), вы модифицируете и веса self.w. Если вы хотите модифицировать веса,
        # (например, в fit), используйте self.w

    def __loss(self, y, p):
        p = np.clip(p, 1e-10, 1 - 1e-10)
        return -np.sum(y * np.log(p) + (1 - y) * np.log(1 - p))

### Проверка градиента на примерах

In [ ]:
m = MyLogisticRegression()
X = np.array([[1, 3, 4], [1, -5, 6], [-3, 5, 3]])
X = np.concatenate((np.ones((X.shape[0], 1)), X), axis=1)
y = np.array([1, 0, 1])
preds = np.array([.55, .22, .85])
grads = m.get_grad(X, y, preds)
assert np.allclose(grads, np.array([-0.38,  0.22, -3.2 , -0.93])), "Что-то не так!"

In [ ]:
np.random.seed(42)
m = MyLogisticRegression()
X = np.random.rand(100,3)
y = np.random.randint(0, 1, size=(100,))
preds = np.random.rand(100)
grads = m.get_grad(X, y, preds)
assert np.allclose(grads, np.array([23.8698149, 25.27049356, 24.4139452])), "Что-то не так!"

## 4. Логистическая регрессия с $l_1$ и $l_2$ регуляризацией (elastic net)

Модель отличается от обычной логистической регрессии только функцией потерь, поэтому `MyElasticLogisticRegression` наследуется от `MyLogisticRegression` и переопределяет методы `get_grad()` и `__loss()`.

Регуляризаторы и их производные:

$$L_1 (w) = \alpha \sum_{j=1}^{n}|w_j|, \qquad L_2 (w) = \beta\sum_{j=1}^{n}w_j^2$$

$$\frac{\partial L_1}{\partial w_j} = \alpha \cdot \mathrm{sign}(w_j), \qquad \frac{\partial L_2}{\partial w_j} = 2\beta w_j$$

Нулевая координата вектора весов — это bias, в регуляризацию он не входит.

In [ ]:
class MyElasticLogisticRegression(MyLogisticRegression):
    def __init__(self, l1_coef, l2_coef):
        self.l1_coef = l1_coef
        self.l2_coef = l2_coef
        self.w = None

    def __loss(self, y, p):
        # YOUR CODE:

    def get_grad(self, X_batch, y_batch, predictions):
        """
        Принимает на вход X_batch с уже добавленной колонкой единиц.
        Выдаёт градиент функции потерь логистической регрессии с регуляризаторами
        как сумму градиентов функции потерь на всех объектах батча + регуляризационное слагаемое
        ВНИМАНИЕ! Нулевая координата вектора весов -- это BIAS, а не вес признака.
        Bias в регуляризационные слагаемые не входит. Также не нужно ДЕЛИТЬ ГРАДИЕНТ НА РАЗМЕР БАТЧА:
        нас интересует не среднее, а сумма.

        Выход -- вектор-столбец градиентов для каждого веса (np.array[n_features + 1])
        """


        grad_basic = # YOUR CODE: компонент градиента из логрегрессии. Обнулять bias-компоненту этой составляющей градиента не нужно!

        grad_l1 = # YOUR CODE: компонент градиента из l_1-регуляризации (не забудьте обнулить компоненту с bias)
        grad_l2 = # YOUR CODE: компонент градиента из l_2-регуляризации (не забудьте обнулить компоненту с bias)

        # Обнулять bias-компоненту вектора весов не нужно!

        assert grad_l1[0] == grad_l2[0] == 0, "Bias в регуляризационные слагаемые не входит!"
        assert grad_basic.shape == grad_l1.shape == grad_l2.shape == (X_batch.shape[1],) , "Градиенты должны быть столбцом из k_features + 1 элементов"

        return grad_basic + grad_l1 + grad_l2

### Проверка градиента с регуляризацией

In [ ]:
m = MyElasticLogisticRegression(.2,.2)
X = np.array([[1, 3, 4], [1, -5, 6], [-3, 5, 3]])
X = np.concatenate((np.ones((X.shape[0], 1)), X), axis=1)
y = np.array([1, 0, 1])
preds = np.array([.55, .22, .85])
m.w = np.array([1,1,1,1])
grads = m.get_grad(X, y, preds)
assert np.allclose(grads, np.array([-0.38,  0.82, -2.6 , -0.33])), "Что-то не так!"


In [ ]:
np.random.seed(42)
m = MyElasticLogisticRegression(.2, .2)
X = np.random.rand(100,3)
X = np.concatenate((np.ones((X.shape[0], 1)), X), axis=1)
y = np.random.randint(0, 1, size=(100,))
preds = np.random.rand(100)
m.w = np.array([1,1,1,1])
grads = m.get_grad(X, y, preds)
assert np.allclose(grads, np.array([49.11489408, 24.4698149, 25.87049356, 25.0139452])), "Что-то не так!"


## 5. Проверка модели

### Синтетические данные

Сначала проверяю модель на искусственных данных (`make_blobs`, два кластера):

In [ ]:
from sklearn.datasets import make_blobs
X, y = make_blobs(n_samples=1000, centers=[[-2,0.5],[3,-0.5]], cluster_std=1, random_state=42)

colors = ("red", "green")
colored_y = np.zeros(y.size, dtype=str)

for i, cl in enumerate([0,1]):
    colored_y[y.ravel() == cl] = str(colors[i])

plt.figure(figsize=(15,10))
plt.scatter(X[:, 0], X[:, 1], c=colored_y)
plt.grid()
plt.show();

In [ ]:
clf = MyElasticLogisticRegression(0.1, 0.1)
clf.fit(X, y, epochs=1000)
w = clf.get_weights()

In [ ]:
from matplotlib.colors import ListedColormap

plt.figure(figsize=(15,8))

eps = 0.1
xx, yy = np.meshgrid(np.linspace(np.min(X[:,0]) - eps, np.max(X[:,0]) + eps, 200),
                     np.linspace(np.min(X[:,1]) - eps, np.max(X[:,1]) + eps, 200))
Z = clf.predict(np.c_[xx.ravel(), yy.ravel()])
Z = Z.reshape(xx.shape)
cmap_light = ListedColormap(['#FFAAAA', '#AAFFAA'])
plt.pcolormesh(xx, yy, Z, cmap=cmap_light)

plt.scatter(X[:, 0], X[:, 1], c=colored_y)

### Цифры MNIST

Теперь проверяю модель на MNIST, классическом датасете рукописных цифр (чёрно-белые изображения 28×28).

Классификатор бинарный, поэтому оставляю только изображения нулей и единиц. Каждое изображение уже представлено вектором из 784 пикселей.

In [ ]:
data = pd.read_csv('./train.csv')
# MNIST в формате CSV: файл в репозиторий не включён, положите train.csv рядом с ноутбуком
data.head()

Первая колонка таблицы — метка (цифра на картинке), следующие 784 колонки — значения пикселей от 0 до 255. Изображения уже превращены в векторы, поэтому `reshape` не нужен.

In [ ]:
X = data.iloc[:, 1:]
y = data.iloc[:, 0]

# Выберем только картинки, где изображен 0 и 1
X = X[(y == 0) | (y == 1)]
y = y[(y == 0) | (y == 1)]

Качество оцениваю кросс-валидацией (метрика accuracy).

In [ ]:
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# TODO: pipeline из StandardScaler и MyElasticLogisticRegression,
# оценка точности через cross_val_score (scoring='accuracy')


print(f"Mean accuracy of Logistic Regression for two classes is {mean_accuracy}")